# Notebook 20
## Protein ProtBERT Downstream Classifiers

Trains and evaluates classical ML classifiers on top of the frozen ProtBERT
embeddings produced by notebook 19.

**Task:** Pfam family classification (10-class)
**Embedding dim:** 1024 (ProtBERT / BERT-large)

**Models:** Logistic Regression, Linear SVM, Random Forest, XGBoost

**Split:** identical `train_test_split(random_state=SEED, test_size=0.2, stratify=y)`
used across all prior protein notebooks.

**Metrics:** accuracy, macro precision, macro recall, macro F1, ROC-AUC (OvR macro)

### Outputs
- `reports/protein_protbert_test_results.csv`
- `reports/protein_protbert_models_summary.json`
- `reports/figures/protein_protbert/cm_<model>.png`
- `models/protein/protbert/<model>.pkl`
- `reports/protein_all_paradigms_comparison.csv` (updated)

## 1) Imports

In [1]:
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
)

warnings.filterwarnings('ignore')
np.set_printoptions(suppress=True)
print('imports OK')


imports OK


## 2) Paths, config, seed

In [2]:
ROOT      = Path.cwd().parents[0]
PROCESSED = ROOT / 'data' / 'processed'
REPORTS   = ROOT / 'reports'
FIGURES   = REPORTS / 'figures' / 'protein_protbert'
MODELS    = ROOT / 'models' / 'protein' / 'protbert'
CONFIGS   = ROOT / 'configs'

for p in [REPORTS, FIGURES, MODELS]:
    p.mkdir(parents=True, exist_ok=True)

with open(CONFIGS / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

SEED    = int(cfg['project']['random_seed'])
N_FAM   = int(cfg['protein']['n_families'])
PER_FAM = int(cfg['protein']['per_family'])

random.seed(SEED)
np.random.seed(SEED)
print(f'SEED={SEED}  N_FAM={N_FAM}  PER_FAM={PER_FAM}')


SEED=42  N_FAM=10  PER_FAM=400


## 3) Load ProtBERT embeddings

Load the three parallel arrays saved by notebook 19 in original row order.

In [3]:
sfx = f'top{N_FAM}_per{PER_FAM}'

emb_path    = PROCESSED / f'protein_protbert_embeddings_{sfx}.npy'
labels_path = PROCESSED / f'protein_protbert_labels_{sfx}.npy'
fnames_path = PROCESSED / f'protein_protbert_family_names_{sfx}.npy'

for p in [emb_path, labels_path, fnames_path]:
    assert p.exists(), f'Missing: {p}  -- run notebook 19 first'

X            = np.load(emb_path)
y            = np.load(labels_path)
family_names = np.load(fnames_path, allow_pickle=True)

# Recover class labels from the saved family names
le = LabelEncoder()
le.fit(family_names)
class_names = le.classes_

print(f'X shape:  {X.shape}   dtype: {X.dtype}')
print(f'y shape:  {y.shape}   n_classes: {len(np.unique(y))}')
print(f'Classes:  {class_names}')
assert not np.isnan(X).any() and not np.isinf(X).any()
print('Integrity checks passed.')


X shape:  (2293, 1024)   dtype: float32
y shape:  (2293,)   n_classes: 10
Classes:  ['PF00001' 'PF00046' 'PF00069' 'PF00071' 'PF00076' 'PF00096' 'PF01352'
 'PF07686' 'PF12796' 'PF13853']
Integrity checks passed.


## 4) Train / test split

Same parameters as all prior protein notebooks:
`test_size=0.2, random_state=SEED, stratify=y`.

In [4]:
X_train, X_test, y_train, y_test, fnames_train, fnames_test = train_test_split(
    X, y, family_names,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

print(f'Train: {X_train.shape}')
print(f'Test:  {X_test.shape}')
print('Test class distribution:')
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  class {u} ({class_names[u]}): {c}')


Train: (1834, 1024)
Test:  (459, 1024)
Test class distribution:
  class 0 (PF00001): 58
  class 1 (PF00046): 37
  class 2 (PF00069): 41
  class 3 (PF00071): 25
  class 4 (PF00076): 24
  class 5 (PF00096): 39
  class 6 (PF01352): 70
  class 7 (PF07686): 62
  class 8 (PF12796): 23
  class 9 (PF13853): 80


## 5) Define classifiers

Identical definitions to notebooks 10 and prior protein model notebooks.
ProtBERT embeddings have std ~0.10 (from summary), so scaling matters
for LR and SVM.

In [6]:
models = {}

models['logreg'] = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=5000, solver='lbfgs',
        random_state=SEED,
    )),
])

models['linear_svm_calibrated'] = CalibratedClassifierCV(
    estimator=Pipeline([
        ('scaler', StandardScaler()),
        ('svm', LinearSVC(random_state=SEED, max_iter=5000)),
    ]),
    method='sigmoid', cv=3,
)

models['random_forest'] = RandomForestClassifier(
    n_estimators=400, random_state=SEED, n_jobs=-1,
)

try:
    from xgboost import XGBClassifier
    models['xgboost'] = XGBClassifier(
        n_estimators=600, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, eval_metric='mlogloss',
        tree_method='hist', objective='multi:softprob',
    )
except ImportError:
    print('XGBoost not available, skipping.')

print('Models defined:', list(models.keys()))

Models defined: ['logreg', 'linear_svm_calibrated', 'random_forest', 'xgboost']


## 6) 5-fold cross-validation on training set

In [7]:
CV_FOLDS = 5
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

cv_results = []
for name, clf in models.items():
    print(f'CV: {name} ...', end=' ', flush=True)
    scores = cross_validate(
        clf, X_train, y_train,
        cv=skf,
        scoring=['accuracy', 'f1_macro', 'roc_auc_ovr_weighted'],
        n_jobs=1,
    )
    row = {
        'model':            name,
        'cv_accuracy_mean': scores['test_accuracy'].mean(),
        'cv_accuracy_std':  scores['test_accuracy'].std(),
        'cv_f1_macro_mean': scores['test_f1_macro'].mean(),
        'cv_f1_macro_std':  scores['test_f1_macro'].std(),
        'cv_roc_auc_mean':  scores['test_roc_auc_ovr_weighted'].mean(),
        'cv_roc_auc_std':   scores['test_roc_auc_ovr_weighted'].std(),
    }
    cv_results.append(row)
    print(f"acc={row['cv_accuracy_mean']:.4f}  "
          f"f1_macro={row['cv_f1_macro_mean']:.4f}  "
          f"roc_auc={row['cv_roc_auc_mean']:.4f}")

df_cv = pd.DataFrame(cv_results)
print('\nCV summary:')
print(df_cv[['model','cv_accuracy_mean','cv_f1_macro_mean','cv_roc_auc_mean']].to_string(index=False))


CV: logreg ... acc=0.9886  f1_macro=0.9866  roc_auc=0.9994
CV: linear_svm_calibrated ... acc=0.9929  f1_macro=0.9919  roc_auc=0.9994
CV: random_forest ... acc=0.9460  f1_macro=0.9317  roc_auc=0.9959
CV: xgboost ... acc=0.9618  f1_macro=0.9530  roc_auc=0.9979

CV summary:
                model  cv_accuracy_mean  cv_f1_macro_mean  cv_roc_auc_mean
               logreg          0.988551          0.986551         0.999359
linear_svm_calibrated          0.992911          0.991882         0.999449
        random_forest          0.946021          0.931681         0.995948
              xgboost          0.961835          0.953049         0.997898


## 7) Train on full training set, evaluate on holdout

Multiclass metrics: macro-averaged precision, recall, F1;
ROC-AUC one-vs-rest (OvR) macro average.

In [8]:
def evaluate(name, clf, X_tr, y_tr, X_te, y_te):
    clf.fit(X_tr, y_tr)
    y_pred  = clf.predict(X_te)
    y_proba = clf.predict_proba(X_te)          # (n_samples, n_classes)
    return {
        'model':     name,
        'accuracy':  float(accuracy_score(y_te, y_pred)),
        'precision_macro': float(precision_score(y_te, y_pred,
                                  average='macro', zero_division=0)),
        'recall_macro':    float(recall_score(y_te, y_pred,
                                  average='macro', zero_division=0)),
        'f1_macro':        float(f1_score(y_te, y_pred,
                                  average='macro', zero_division=0)),
        'roc_auc_ovr':     float(roc_auc_score(y_te, y_proba,
                                  multi_class='ovr', average='macro')),
    }, clf, y_pred, y_proba


test_results  = []
fitted_models = {}

for name, clf in models.items():
    print(f'Fitting: {name} ...', end=' ', flush=True)
    metrics, fitted_clf, y_pred, y_proba = evaluate(
        name, clf, X_train, y_train, X_test, y_test)
    test_results.append(metrics)
    fitted_models[name] = (fitted_clf, y_pred, y_proba)
    joblib.dump(fitted_clf, MODELS / f'{name}.pkl')
    print(f"acc={metrics['accuracy']:.4f}  "
          f"f1_macro={metrics['f1_macro']:.4f}  "
          f"roc_auc={metrics['roc_auc_ovr']:.4f}")

df_test = pd.DataFrame(test_results)
print('\nHoldout results:')
print(df_test.to_string(index=False))


Fitting: logreg ... acc=0.9826  f1_macro=0.9834  roc_auc=0.9994
Fitting: linear_svm_calibrated ... acc=0.9913  f1_macro=0.9924  roc_auc=0.9982
Fitting: random_forest ... acc=0.9651  f1_macro=0.9546  roc_auc=0.9982
Fitting: xgboost ... acc=0.9847  f1_macro=0.9833  roc_auc=0.9973

Holdout results:
                model  accuracy  precision_macro  recall_macro  f1_macro  roc_auc_ovr
               logreg  0.982571         0.986539      0.981029  0.983365     0.999389
linear_svm_calibrated  0.991285         0.994656      0.990445  0.992410     0.998212
        random_forest  0.965142         0.959081      0.951454  0.954579     0.998246
              xgboost  0.984749         0.985224      0.981809  0.983341     0.997302


## 8) Save results CSV

In [9]:
results_csv = REPORTS / 'protein_protbert_test_results.csv'
df_test.to_csv(results_csv, index=False)
print('Saved:', results_csv)


Saved: /home/dpratapa/Capstone/reports/protein_protbert_test_results.csv


## 9) Confusion matrices

In [10]:
for name, (fitted_clf, y_pred, _) in fitted_models.items():
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 7))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    )
    disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'ProtBERT / {name}')
    plt.tight_layout()
    fig.savefig(FIGURES / f'cm_{name}.png', dpi=150)
    plt.close(fig)

print('Confusion matrices saved to:', FIGURES)


Confusion matrices saved to: /home/dpratapa/Capstone/reports/figures/protein_protbert


## 10) Summary JSON

In [11]:
df_merged = df_test.merge(df_cv, on='model', how='left')
best_row  = df_test.loc[df_test['roc_auc_ovr'].idxmax()]

summary = {
    'notebook': '20_protein_protbert_models',
    'embedding_source': '19_protein_protbert_embeddings',
    'model': 'Rostlab/prot_bert',
    'embedding_dim': int(X.shape[1]),
    'n_train': int(X_train.shape[0]),
    'n_test':  int(X_test.shape[0]),
    'n_classes': int(len(class_names)),
    'class_names': class_names.tolist(),
    'cv_folds': CV_FOLDS,
    'seed': SEED,
    'best_model': {
        'name':       str(best_row['model']),
        'roc_auc_ovr':float(best_row['roc_auc_ovr']),
        'f1_macro':   float(best_row['f1_macro']),
        'accuracy':   float(best_row['accuracy']),
    },
    'all_results': df_merged.to_dict(orient='records'),
    'model_files': {n: str(MODELS / f'{n}.pkl') for n in models},
    'timestamp': pd.Timestamp.now().isoformat(),
}

summary_path = REPORTS / 'protein_protbert_models_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary saved to:', summary_path)
print(f"Best model: {summary['best_model']['name']}  "
      f"ROC-AUC={summary['best_model']['roc_auc_ovr']:.4f}  "
      f"F1-macro={summary['best_model']['f1_macro']:.4f}")


Summary saved to: /home/dpratapa/Capstone/reports/protein_protbert_models_summary.json
Best model: logreg  ROC-AUC=0.9994  F1-macro=0.9834


## 11) Full protein paradigm comparison

Collects results from baseline, CNN, hybrid, ESM-2, and ProtBERT notebooks
into a single comparison table sorted by accuracy.

**Note on metrics:** baseline/CNN/hybrid notebooks report binary-style metrics
since they use the same Pfam 10-class task but may report accuracy differently.
We align on `accuracy` and `f1_macro` / `f1` as the primary comparison columns.

In [12]:
# ESM-2 results CSV from notebook 14
# Baseline protein results from notebook 10
result_files = {
    'baseline': REPORTS / 'protein_baseline_test_results.csv',
    'cnn':      REPORTS / 'protein_seq_cnn_test_results.csv',
    'hybrid':   REPORTS / 'protein_hybrid_test_results.csv',
    'esm2':     REPORTS / 'protein_esm2_test_results.csv',
    'protbert': REPORTS / 'protein_protbert_test_results.csv',
}

frames = []
for paradigm, path in result_files.items():
    if path.exists():
        df_tmp = pd.read_csv(path)
        df_tmp.insert(0, 'paradigm', paradigm)
        frames.append(df_tmp)
    else:
        print(f'Not found (skipping): {path}')

if frames:
    df_all = pd.concat(frames, ignore_index=True)
    # Normalise column names: some notebooks use f1, others f1_macro
    if 'f1' in df_all.columns and 'f1_macro' not in df_all.columns:
        df_all['f1_macro'] = df_all['f1']
    if 'roc_auc' in df_all.columns and 'roc_auc_ovr' not in df_all.columns:
        df_all['roc_auc_ovr'] = df_all['roc_auc']
    cols = [c for c in ['paradigm','model','accuracy','f1_macro','roc_auc_ovr']
            if c in df_all.columns]
    df_all = df_all[cols].sort_values('accuracy', ascending=False)
    print(df_all.to_string(index=False))

    comparison_csv = REPORTS / 'protein_all_paradigms_comparison.csv'
    df_all.to_csv(comparison_csv, index=False)
    print('\nComparison saved to:', comparison_csv)


Not found (skipping): /home/dpratapa/Capstone/reports/protein_baseline_test_results.csv
Not found (skipping): /home/dpratapa/Capstone/reports/protein_seq_cnn_test_results.csv
Not found (skipping): /home/dpratapa/Capstone/reports/protein_hybrid_test_results.csv
Not found (skipping): /home/dpratapa/Capstone/reports/protein_esm2_test_results.csv
paradigm                 model  accuracy  f1_macro  roc_auc_ovr
protbert linear_svm_calibrated  0.991285  0.992410     0.998212
protbert               xgboost  0.984749  0.983341     0.997302
protbert                logreg  0.982571  0.983365     0.999389
protbert         random_forest  0.965142  0.954579     0.998246

Comparison saved to: /home/dpratapa/Capstone/reports/protein_all_paradigms_comparison.csv


## Next

All transformer embedding notebooks are complete. The full pipeline is:

**DNA:** baseline -> CNN -> hybrid -> DNABERT-2 -> Nucleotide Transformer

**Protein:** baseline -> CNN -> hybrid -> ESM-2 -> ProtBERT

Next steps:
- Produce final consolidated comparison figures (bar charts, tables)
- Write interpretation section
- Build Streamlit app for interactive sequence classification and comparison

In [13]:
from pathlib import Path
reports = Path('../reports')
for f in sorted(reports.glob('protein_*')):
    print(f.name)

protein_all_paradigms_comparison.csv
protein_baseline_cv_results_top10_per400.csv
protein_baseline_features_summary.json
protein_baseline_features_summary_top10_per400.json
protein_baseline_models_summary_top10_per400.json
protein_baseline_test_results_top10_per400.csv
protein_eda_summary.json
protein_hybrid_summary_top10_per400.json
protein_hybrid_test_results_top10_per400.csv
protein_ingest_summary.json
protein_protbert_embeddings_summary.json
protein_protbert_models_summary.json
protein_protbert_test_results.csv
protein_seq_cnn_summary_top10_per400.json
protein_seq_cnn_test_results_top10_per400.csv
